In [2]:
import subprocess
import sys
subprocess.check_call([sys.executable,'-m','pip','install',
    'qiskit==2.2.3','qiskit-machine-learning==0.9.0','numpy==2.2.6',
    'scipy==1.15.3','scikit-learn==1.6.1','pandas==2.2.3',
    'matplotlib==3.10.1','flask==3.1.0'])

0

In [12]:
import numpy as np
x=np.array([.6,.8,.3,.4])
score=x[1]+(1-x[2])
print('규칙 점수:',round(float(score),4))
print('교육용 그룹:',int(score>1))

규칙 점수: 1.5
교육용 그룹: 1


In [13]:
import time
from pathlib import Path
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from qiskit.circuit.library import z_feature_map, zz_feature_map, pauli_feature_map
from qiskit.circuit.library import real_amplitudes, efficient_su2, n_local, pauli_two_design
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.primitives import QMLSampler, QMLEstimator
from qiskit_machine_learning.neural_networks import SamplerQNN, EstimatorQNN
from qiskit_machine_learning.algorithms.classifiers import VQC

def parity(integer):
    return int(integer).bit_count() % 2

def circuits(n, fmap_name='Z', ansatz_name='RealAmplitudes', reps=1):
    if fmap_name == 'Z':
        fmap = z_feature_map(n, reps=1)
    elif fmap_name == 'ZZ':
        fmap = zz_feature_map(n, reps=1, entanglement='linear')
    elif fmap_name == 'Pauli':
        fmap = pauli_feature_map(n, reps=1, paulis=['Z','XX'], entanglement='linear')
    else:
        raise ValueError(fmap_name)
    if ansatz_name == 'RealAmplitudes':
        ansatz = real_amplitudes(n, reps=reps, entanglement='linear')
    elif ansatz_name == 'EfficientSU2':
        ansatz = efficient_su2(n, reps=reps, entanglement='linear')
    elif ansatz_name == 'TwoLocal':
        ansatz = n_local(n, rotation_blocks=['ry','rz'], entanglement_blocks='cz', reps=reps, entanglement='linear')
    elif ansatz_name == 'PauliTwoDesign':
        ansatz = pauli_two_design(n, reps=reps, seed=33)
    else:
        raise ValueError(ansatz_name)
    return fmap, ansatz

def fit_model(kind, train_x, train_y, fmap_name='Z', ansatz_name='RealAmplitudes', maxiter=40, reps=1):
    fmap, ansatz = circuits(train_x.shape[1], fmap_name, ansatz_name, reps)
    initial = np.linspace(0.1,0.7,ansatz.num_parameters)
    history = []
    def optimizer(fun, x0, jac=None, bounds=None):
        def tracked(weights):
            value = float(fun(weights))
            history.append(value)
            return value
        return minimize(tracked, x0, method='COBYLA', options={'maxiter':maxiter, 'rhobeg':0.3})
    started = time.perf_counter()
    if kind == 'VQC':
        model = VQC(feature_map=fmap, ansatz=ansatz, sampler=QMLSampler(shots=None),
                    interpret=parity, output_shape=2, initial_point=initial, optimizer=optimizer)
        model.fit(train_x, train_y)
        def predict_probability(x):
            return np.asarray(model.neural_network.forward(x, model.weights))[:,1]
        weights = model.weights
    else:
        circuit = fmap.compose(ansatz)
        if kind == 'SQNN':
            qnn = SamplerQNN(circuit=circuit, sampler=QMLSampler(shots=None),
                             input_params=list(fmap.parameters), weight_params=list(ansatz.parameters),
                             interpret=parity, output_shape=2)
            def probability(x,w):
                return np.asarray(qnn.forward(x,w))[:,1]
            start = initial
        elif kind == 'EQNN':
            qnn = EstimatorQNN(circuit=circuit, estimator=QMLEstimator(default_precision=0),
                              observables=SparsePauliOp('Z'*train_x.shape[1]),
                              input_params=list(fmap.parameters), weight_params=list(ansatz.parameters),
                              default_precision=0)
            def probability(x,w):
                expectation = np.asarray(qnn.forward(x,w[:-2])).ravel()
                # 마지막 두 값은 고전 출력층의 가중치와 편향입니다.
                score = np.clip(w[-2]*expectation+w[-1],-40,40)
                return 1/(1+np.exp(-score))
            start = np.r_[initial,-1.0,0.0]
        else:
            raise ValueError(kind)
        def loss(w):
            p1 = np.clip(probability(train_x,w),1e-10,1-1e-10)
            value = float(-np.mean(train_y*np.log2(p1)+(1-train_y)*np.log2(1-p1)))
            return value
        result = optimizer(loss,start)
        weights = result.x
        def predict_probability(x):
            return probability(x,weights)
        model = None
    print(kind, fmap_name, ansatz_name, '학습 평가 수:',len(history),
          '최초/마지막 기록 손실:',round(history[0],6),round(history[-1],6))
    return predict_probability, model, {'parameters':len(weights),'depth':fmap.compose(ansatz).depth(),
                                       'seconds':time.perf_counter()-started,'history':history}

def report(name,y,p1):
    p1 = np.asarray(p1).ravel()
    assert p1.shape == y.shape and np.isfinite(p1).all()
    assert np.all((p1>=-1e-8)&(p1<=1+1e-8))
    predicted = (p1>=0.5).astype(int)
    result = {'model':name, 'accuracy':accuracy_score(y,predicted),
              'precision':precision_score(y,predicted,zero_division=0),
              'recall':recall_score(y,predicted,zero_division=0),
              'f1':f1_score(y,predicted,zero_division=0)}
    print({k:round(float(v),4) if k!='model' else v for k,v in result.items()})
    print('혼동행렬 [[TN,FP],[FN,TP]]:',confusion_matrix(y,predicted,labels=[0,1]).tolist())
    return result

def split_fixed(X,y):
    # 각 클래스에서 원래 행 순서의 네 번째마다 시험 표본으로 사용합니다.
    test = np.concatenate([np.flatnonzero(y==c)[3::4] for c in [0,1]])
    train = np.setdiff1d(np.arange(len(y)),test)
    assert not np.intersect1d(train,test).size
    assert len(np.unique(y[train]))==len(np.unique(y[test]))==2
    return train,test

print('버전:',{k:metadata.version(k) for k in ['qiskit','qiskit-machine-learning','numpy','scipy','scikit-learn']})

X = np.array([[0.2, 0.2, 0.8, 0.8],
 [0.8, 0.8, 0.2, 0.2],
 [0.4, 0.3, 0.7, 0.6],
 [0.6, 0.7, 0.3, 0.4],
 [0.7, 0.1, 0.9, 0.3],
 [0.3, 0.9, 0.1, 0.7],
 [0.5, 0.4, 0.8, 0.5],
 [0.5, 0.8, 0.4, 0.5],
 [0.1, 0.2, 0.6, 0.9],
 [0.9, 0.6, 0.2, 0.1],
 [0.8, 0.3, 0.6, 0.2],
 [0.2, 0.6, 0.3, 0.8],
 [0.3, 0.4, 0.7, 0.7],
 [0.7, 0.7, 0.4, 0.3],
 [0.6, 0.2, 0.7, 0.4],
 [0.4, 0.7, 0.2, 0.6],
 [0.9, 0.1, 0.8, 0.1],
 [0.1, 0.8, 0.1, 0.9],
 [0.4, 0.3, 0.5, 0.6],
 [0.6, 0.5, 0.3, 0.4],
 [0.2, 0.4, 0.6, 0.8],
 [0.8, 0.6, 0.4, 0.2],
 [0.7, 0.2, 0.5, 0.3],
 [0.3, 0.5, 0.2, 0.7]], dtype=float)
y = (X[:,1]+(1-X[:,2])>1).astype(int)
train,test = split_fixed(X,y)
scaler = MinMaxScaler(feature_range=(0,1),clip=True).fit(X[train])
train_x,test_x = scaler.transform(X[train]),scaler.transform(X[test])
print('학습/시험 크기:',train_x.shape,test_x.shape)
print('학습/시험 라벨 수:',np.bincount(y[train]).tolist(),np.bincount(y[test]).tolist())
import joblib
predict,model,info=fit_model('VQC',train_x,y[train],'Z','PauliTwoDesign',maxiter=80)
report('PreferenceVQC',y[test],predict(test_x))
model.to_dill('preference_model.dill')
joblib.dump(scaler,'preference_scaler.joblib')
restored=VQC.from_dill('preference_model.dill')
restored_scaler=joblib.load('preference_scaler.joblib')
restored_p=np.asarray(restored.neural_network.forward(test_x,restored.weights))[:,1]
assert np.allclose(predict(test_x),restored_p)
sample=np.array([[.6,.8,.3,.4]])
p1=float(restored.neural_network.forward(restored_scaler.transform(sample),restored.weights)[0,1])
print('저장/복원 확률 일치:',True)
print('단건 P(1)/그룹:',round(p1,6),'그룹 1' if p1>=.5 else '그룹 0')
print('시험 P(1):',np.round(restored_p,4).tolist())

버전: {'qiskit': '2.2.3', 'qiskit-machine-learning': '0.9.0', 'numpy': '2.2.6', 'scipy': '1.15.3', 'scikit-learn': '1.6.1'}
학습/시험 크기: (18, 4) (6, 4)
학습/시험 라벨 수: [9, 9] [3, 3]
VQC Z PauliTwoDesign 학습 평가 수: 80 최초/마지막 기록 손실: 0.995347 0.799833
{'model': 'PreferenceVQC', 'accuracy': 0.6667, 'precision': 1.0, 'recall': 0.3333, 'f1': 0.5}
혼동행렬 [[TN,FP],[FN,TP]]: [[3, 0], [2, 1]]
저장/복원 확률 일치: True
단건 P(1)/그룹: 0.508153 그룹 1
시험 P(1): [0.3828, 0.4242, 0.4861, 0.5201, 0.4754, 0.3331]


시험 혼동행렬 [[3,0],[2,1]]은 그룹 0의 3건을 맞히고 그룹 1의 3건 중 1건을 맞힌 결과입니다. F1은 0.5입니다. 저장·복원 확률 일치가 True이면 복원 과정에서 이 시험 입력들의 모델 출력이 유지됐음을 뜻합니다. 단건 출력 약 0.508153은 그룹 1로 분류되며 임계값에 가깝습니다. 이 출력의 의미는 교육용 그룹 분류입니다.